In [ ]:
import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)

assert gpus, "GPU tidak terdeteksi!"

tf.config.experimental.set_memory_growth(gpus[0], True)

with tf.device("/GPU:0"):
    a = tf.ones((256, 256))
    result = tf.matmul(a, a)

print("Execution device:", result.device)
print("GPU notebook test: PASSED")

## 2. Dataset Manifest Validation

Load the immutable Week 1 baseline manifests and validate
training and validation sample counts before constructing
the TensorFlow data pipeline.

The test split is reserved for final evaluation.

In [ ]:
from pathlib import Path
import csv
from collections import Counter

# Find repository root
current = Path.cwd().resolve()

repo_root = next(
    (p for p in [current, *current.parents]
     if (p / "ml/data/splits/v1/train.csv").is_file()),
    None
)

assert repo_root is not None, "EcoVision repository not found"

split_dir = repo_root / "ml/data/splits/v1"

def load_manifest(name):
    path = split_dir / f"{name}.csv"
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

train_rows = load_manifest("train")
val_rows = load_manifest("val")

print("Repository:", repo_root)
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Columns:", list(train_rows[0].keys()))

print("\nTraining distribution:")
print(Counter(row["label"] for row in train_rows))

print("\nValidation distribution:")
print(Counter(row["label"] for row in val_rows))

assert len(train_rows) == 1766
assert len(val_rows) == 378

print("\nManifest count validation: PASSED")

## 3. Dataset Image Path Validation

Verify that training and validation images exist and can be
resolved from the local TrashNet directory.

The test split remains untouched.

In [ ]:
# Locate TrashNet image directory
raw_dir = repo_root / "ml/data/raw"

candidate_roots = [
    raw_dir,
    raw_dir / "dataset-resized",
    raw_dir / "trashnet",
    raw_dir / "trashnet/dataset-resized",
]

all_rows = train_rows + val_rows

print("Example manifest paths:")
for row in train_rows[:5]:
    print(" -", row["path"])

print("\nChecking candidate directories:")

valid_roots = []

for root in candidate_roots:
    missing = sum(
        not (root / row["path"]).is_file()
        for row in all_rows
    )

    print(
        f"{root}: "
        f"{len(all_rows) - missing}/{len(all_rows)} found"
    )

    if missing == 0:
        valid_roots.append(root)

assert valid_roots, (
    "No directory resolves all training and validation images"
)

image_root = valid_roots[0]

print("\nSelected image root:", image_root)
print("Image path validation: PASSED")

## 4. Image Integrity Verification

Verify representative images from each class against the
SHA-256 hashes recorded in the Week 1 dataset manifests.

This check prevents accidentally selecting an incorrect
image directory.

In [ ]:
import hashlib

CLASS_NAMES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

# Ambil 1 gambar dari tiap kelas pada train dan validation
sample_rows = [
    next(row for row in rows if row["label"] == label)
    for rows in (train_rows, val_rows)
    for label in CLASS_NAMES
]

def calculate_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Bandingkan setiap kandidat direktori
matching_roots = []

for root in valid_roots:
    mismatches = []

    for row in sample_rows:
        image_path = root / row["path"]
        actual_hash = calculate_sha256(image_path)

        if actual_hash != row["sha256"]:
            mismatches.append(row["path"])

    print(f"\nDirectory: {root}")
    print(f"Verified: {len(sample_rows) - len(mismatches)}/{len(sample_rows)}")

    if not mismatches:
        matching_roots.append(root)
    else:
        print("Mismatched files:", mismatches)

assert image_root in matching_roots, (
    "Selected image root does not match Week 1 hashes"
)

print("\nSample image integrity: PASSED")

In [ ]:
# Full integrity check — train and validation only

all_rows = train_rows + val_rows
mismatches = []

for row in all_rows:
    image_path = image_root / row["path"]

    if calculate_sha256(image_path) != row["sha256"]:
        mismatches.append(row["path"])

print("Images checked:", len(all_rows))
print("SHA-256 mismatches:", len(mismatches))

assert not mismatches, f"Integrity errors: {mismatches[:5]}"

# Prevent exact duplicate and grouped-image leakage
train_hashes = {r["sha256"] for r in train_rows}
val_hashes = {r["sha256"] for r in val_rows}

train_groups = {r["group_id"] for r in train_rows}
val_groups = {r["group_id"] for r in val_rows}

assert train_hashes.isdisjoint(val_hashes)
assert train_groups.isdisjoint(val_groups)

print("Full image integrity: PASSED")
print("Train/validation leakage checks: PASSED")

## 5. TensorFlow Data Pipeline

Create reproducible training and validation pipelines
using the verified Week 1 dataset manifests.

- Input: 224 × 224 RGB
- Pixel values: float32, range 0–255
- Batch size: 8
- Training: shuffled
- Validation: deterministic order
- No augmentation or normalization in the data pipeline
- Test split remains reserved for final evaluation

In [ ]:
import tensorflow as tf

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 8

CLASS_NAMES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

NUM_CLASSES = len(CLASS_NAMES)

label_to_id = {
    label: idx
    for idx, label in enumerate(CLASS_NAMES)
}

assert set(label_to_id) == {
    row["label"] for row in train_rows + val_rows
}

tf.random.set_seed(SEED)


def decode_and_resize(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    # Resize and convert to float32
    image = tf.image.resize(
        image,
        IMG_SIZE,
        method="bilinear",
        antialias=True
    )

    # Keep original pixel range: 0–255
    image = tf.ensure_shape(image, [224, 224, 3])

    return image, label


def create_dataset(rows, training=False):
    paths = [
        str(image_root / row["path"])
        for row in rows
    ]

    labels = tf.constant(
        [label_to_id[row["label"]] for row in rows],
        dtype=tf.int32
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(rows),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        decode_and_resize,
        num_parallel_calls=2
    )

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    dataset = dataset.prefetch(1)

    return dataset


train_ds = create_dataset(train_rows, training=True)
val_ds = create_dataset(val_rows, training=False)

print("Training batches:", train_ds.cardinality().numpy())
print("Validation batches:", val_ds.cardinality().numpy())
print("Classes:", CLASS_NAMES)
print("tf.data pipeline: CREATED")

In [ ]:
# Dataset pipeline smoke test

images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Image dtype:", images.dtype)
print("Label batch shape:", labels.shape)
print("Label dtype:", labels.dtype)
print("Minimum pixel:", float(tf.reduce_min(images)))
print("Maximum pixel:", float(tf.reduce_max(images)))
print("Sample label IDs:", labels.numpy())

assert images.shape == (8, 224, 224, 3)
assert images.dtype == tf.float32
assert labels.dtype == tf.int32

tf.debugging.assert_all_finite(
    images,
    "Images contain NaN or infinity"
)

assert float(tf.reduce_min(images)) >= 0
assert float(tf.reduce_max(images)) <= 255

assert bool(tf.reduce_all(labels >= 0))
assert bool(tf.reduce_all(labels < NUM_CLASSES))

print("\nDataset pipeline smoke test: PASSED")

## 6. Training-only Data Augmentation

Apply conservative image augmentation to the training
dataset only. Validation images remain unchanged.

Augmentation is kept outside the inference model to
simplify future TensorFlow Lite export.

In [ ]:
# Training-only augmentation

from tensorflow.keras import layers

augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=SEED
        ),
        layers.RandomRotation(
            0.04,
            fill_mode="reflect",
            seed=SEED + 1
        ),
        layers.RandomZoom(
            0.08,
            fill_mode="reflect",
            seed=SEED + 2
        ),
    ],
    name="training_augmentation"
)

def augment_batch(images, labels):
    augmented = augmentation(images, training=True)
    augmented = tf.clip_by_value(augmented, 0.0, 255.0)
    return augmented, labels

train_aug_ds = train_ds.map(
    augment_batch,
    num_parallel_calls=1
).prefetch(1)

# Validation remains unchanged
validation_ds = val_ds

aug_images, aug_labels = next(iter(train_aug_ds))

assert aug_images.shape == (8, 224, 224, 3)
assert aug_images.dtype == tf.float32
assert tf.reduce_min(aug_images).numpy() >= 0
assert tf.reduce_max(aug_images).numpy() <= 255

print("Augmented batch shape:", aug_images.shape)
print("Augmented pixel range:",
      float(tf.reduce_min(aug_images)),
      float(tf.reduce_max(aug_images)))

print("Training augmentation: PASSED")

## 7. MobileNetV2 Baseline Architecture

Build a six-class image classifier using ImageNet-pretrained
MobileNetV2 as a frozen feature extractor.

Input preprocessing is embedded in the model to preserve
the EcoVision inference contract.

In [ ]:
# MobileNetV2 baseline architecture

from tensorflow.keras import Model, layers

tf.keras.utils.set_random_seed(SEED)

# Pretrained feature extractor
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    alpha=1.0
)

base_model.trainable = False

# EcoVision model contract
inputs = layers.Input(
    shape=(224, 224, 3),
    dtype=tf.float32,
    name="image_rgb"
)

# Keep preprocessing inside the exported model
x = layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1.0,
    name="mobilenetv2_rescaling"
)(inputs)

# Frozen ImageNet feature extractor
x = base_model(x, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32",
    name="waste_probabilities"
)(x)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="ecovision_mobilenetv2_baseline"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

model.summary()

In [ ]:
# Model forward-pass smoke test

sample_input = images[:1]

predictions = model(
    sample_input,
    training=False
)

print("Input shape:", sample_input.shape)
print("Output shape:", predictions.shape)
print("Output probabilities:", predictions.numpy())
print("Probability sum:",
      float(tf.reduce_sum(predictions)))

assert predictions.shape == (1, NUM_CLASSES)

tf.debugging.assert_near(
    tf.reduce_sum(predictions, axis=1),
    tf.ones((1,), dtype=tf.float32),
    atol=1e-5
)

print("\nModel forward-pass test: PASSED")

## 8. Baseline Training — Frozen MobileNetV2

Train the classification head while keeping the ImageNet
pretrained MobileNetV2 backbone frozen.

Training configuration:
- Optimizer: Adam
- Learning rate: 0.001
- Batch size: 8
- Maximum epochs: 12
- EarlyStopping: monitor validation loss
- ModelCheckpoint: save the best validation model
- CSVLogger: record experiment metrics

The test dataset remains reserved for final evaluation.

In [ ]:
from pathlib import Path
from datetime import datetime
import tensorflow as tf

EPOCHS = 12

# Experiment output: intentionally excluded from Git
run_id = datetime.now().strftime(
    "mobilenetv2_frozen_%Y%m%d_%H%M%S"
)

experiment_dir = (
    repo_root / "ml" / "reports" / "generated"
    / "week2" / run_id
)

experiment_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    experiment_dir / "best_model.keras"
)

history_path = (
    experiment_dir / "training_history.csv"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=4,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(history_path)
    )
]

assert base_model.trainable is False

print("Experiment:", run_id)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Checkpoint:", best_model_path)
print("History:", history_path)
print("Training configuration: READY")

In [ ]:
# Frozen MobileNetV2 baseline training

history = model.fit(
    train_aug_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed.")
print("Epochs executed:", len(history.history["loss"]))
print("Best validation loss:", min(history.history["val_loss"]))
print("Best validation accuracy:", max(history.history["val_accuracy"]))
print("Best model:", best_model_path)

## 9. Baseline Checkpoint Verification

Verify the saved best-validation-loss model and inspect
the frozen-backbone training history.

The test split remains untouched.

In [ ]:
import csv
import tensorflow as tf

# Verify experiment files
assert best_model_path.is_file(), "Checkpoint not found"
assert history_path.is_file(), "Training history not found"

# Read training history
with open(history_path, "r", newline="") as f:
    records = list(csv.DictReader(f))

assert records, "Training history is empty"

best_loss_idx = min(
    range(len(records)),
    key=lambda i: float(records[i]["val_loss"])
)

best_acc_idx = max(
    range(len(records)),
    key=lambda i: float(records[i]["val_accuracy"])
)

print("Epochs recorded:", len(records))
print("Best loss epoch:", best_loss_idx + 1)
print("Best accuracy epoch:", best_acc_idx + 1)

print(
    "Minimum validation loss:",
    records[best_loss_idx]["val_loss"]
)

print(
    "Maximum validation accuracy:",
    records[best_acc_idx]["val_accuracy"]
)

# Reload model saved by ModelCheckpoint
best_model = tf.keras.models.load_model(
    str(best_model_path)
)

# Evaluate on validation only
val_metrics = best_model.evaluate(
    validation_ds,
    return_dict=True,
    verbose=1
)

print("\nReloaded checkpoint metrics:")
for name, value in val_metrics.items():
    print(f"{name}: {value:.6f}")

print("\nCheckpoint verification: COMPLETED")

## 10. Baseline Training Curves

Visualize training and validation accuracy/loss across
all recorded epochs.

Use the curves to inspect convergence, possible overfitting,
and whether further fine-tuning is justified.

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, len(records) + 1))

train_acc = [float(r["accuracy"]) for r in records]
val_acc = [float(r["val_accuracy"]) for r in records]

train_loss = [float(r["loss"]) for r in records]
val_loss = [float(r["val_loss"]) for r in records]

best_epoch = best_loss_idx + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Accuracy
axes[0].plot(epochs, train_acc, marker="o",
             label="Training Accuracy")
axes[0].plot(epochs, val_acc, marker="o",
             label="Validation Accuracy")
axes[0].axvline(best_epoch, linestyle="--",
                color="gray", label=f"Best Epoch: {best_epoch}")
axes[0].set_title("MobileNetV2 — Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs, train_loss, marker="o",
             label="Training Loss")
axes[1].plot(epochs, val_loss, marker="o",
             label="Validation Loss")
axes[1].axvline(best_epoch, linestyle="--",
                color="gray", label=f"Best Epoch: {best_epoch}")
axes[1].set_title("MobileNetV2 — Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

plot_path = experiment_dir / "training_curves.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")

plt.show()

print("Training curves saved:", plot_path)
print("Best checkpoint epoch:", best_epoch)

## 11. Validation Classification Report

Evaluate the best frozen MobileNetV2 checkpoint using
the validation split only.

Metrics:
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score
- Per-class Precision, Recall, and F1-score

The test split remains sealed.

In [ ]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Ground-truth labels from validation manifest
y_true = np.array(
    [label_to_id[row["label"]] for row in val_rows],
    dtype=np.int32
)

# Predict using the verified best checkpoint
# validation_ds has no shuffling
y_prob = best_model.predict(
    validation_ds,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)

# Verify prediction contract
assert y_prob.shape == (378, NUM_CLASSES)
assert len(y_true) == len(y_pred)
assert np.all(np.isfinite(y_prob))

np.testing.assert_allclose(
    y_prob.sum(axis=1),
    np.ones(len(y_prob)),
    atol=1e-5
)

# Evaluation metrics
accuracy = accuracy_score(y_true, y_pred)

macro_precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print("=== Validation Metrics ===")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Macro Precision : {macro_precision:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")
print(f"Macro F1-score  : {macro_f1:.4f}")

print("\n=== Per-Class Classification Report ===")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

print("Validation classification report: COMPLETED")

## 12. Validation Confusion Matrix

Analyze class-level prediction errors using the best
MobileNetV2 baseline checkpoint.

Generate:
- Confusion matrix with raw counts
- Row-normalized confusion matrix
- Most frequent misclassification pairs

Only validation data is used.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Ground truth and predictions from previous cell
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES))
)

assert cm.shape == (NUM_CLASSES, NUM_CLASSES)
assert cm.sum() == len(val_rows)
assert np.trace(cm) == np.sum(y_true == y_pred)

print("=== Confusion Matrix ===")
print(cm)

print("\nCorrect predictions:", np.trace(cm))
print("Incorrect predictions:", cm.sum() - np.trace(cm))

# 1. Raw confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45
)

ax.set_title("MobileNetV2 Baseline — Confusion Matrix")

plt.tight_layout()

cm_path = experiment_dir / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()

# 2. Row-normalized confusion matrix
cm_normalized = (
    cm.astype(float) /
    cm.sum(axis=1, keepdims=True)
)

fig, ax = plt.subplots(figsize=(8, 7))

disp_normalized = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized,
    display_labels=CLASS_NAMES
)

disp_normalized.plot(
    ax=ax,
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45
)

ax.set_title(
    "MobileNetV2 Baseline — Normalized Confusion Matrix"
)

plt.tight_layout()

normalized_path = (
    experiment_dir / "confusion_matrix_normalized.png"
)

fig.savefig(normalized_path, dpi=150, bbox_inches="tight")
plt.show()

# 3. Most frequent classification errors
errors = []

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            errors.append((
                int(cm[i, j]),
                CLASS_NAMES[i],
                CLASS_NAMES[j]
            ))

errors.sort(reverse=True)

print("\n=== Most Frequent Misclassifications ===")

for count, actual, predicted in errors[:10]:
    print(f"{actual} -> {predicted}: {count} images")

print("\nSaved:", cm_path)
print("Saved:", normalized_path)
print("Confusion matrix evaluation: COMPLETED")

## 13. Targeted Validation Error Analysis

Visually inspect validation images misclassified as:
- Trash incorrectly predicted as another class
- Plastic incorrectly predicted as glass

Record possible visual confusion patterns before
deciding on fine-tuning or class imbalance mitigation.

Do not modify the test dataset.

In [ ]:

# ============================================================
# 13. Targeted Validation Error Analysis
# EcoVision — MobileNetV2 Frozen Baseline
# ============================================================

import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ------------------------------------------------------------
# 1. Select targeted validation errors
# ------------------------------------------------------------

priority_indices = [
    i for i in range(len(val_rows))
    if (
        # Trash incorrectly classified as another class
        (
            y_true[i] == label_to_id["trash"]
            and y_pred[i] != y_true[i]
        )
        or
        # Plastic incorrectly classified as glass
        (
            y_true[i] == label_to_id["plastic"]
            and y_pred[i] == label_to_id["glass"]
        )
    )
]

print("Targeted misclassified images:", len(priority_indices))

assert priority_indices, "No targeted errors found"

trash_errors = sum(
    y_true[i] == label_to_id["trash"]
    for i in priority_indices
)

plastic_to_glass_errors = sum(
    y_true[i] == label_to_id["plastic"]
    for i in priority_indices
)

print("Trash misclassifications:", trash_errors)
print("Plastic -> Glass:", plastic_to_glass_errors)

# ------------------------------------------------------------
# 2. Configure visualization grid
# ------------------------------------------------------------

ncols = 4
nrows = math.ceil(len(priority_indices) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(16, nrows * 3.5),
    squeeze=False
)

axes = axes.flatten()

# ------------------------------------------------------------
# 3. Load and display images using TensorFlow
# ------------------------------------------------------------

for ax, idx in zip(axes, priority_indices):

    row = val_rows[idx]
    image_path = image_root / row["path"]

    # Read image directly using TensorFlow
    image_bytes = tf.io.read_file(str(image_path))

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    ).numpy()

    # Retrieve actual and predicted labels
    true_label = CLASS_NAMES[int(y_true[idx])]
    predicted_label = CLASS_NAMES[int(y_pred[idx])]

    # Predicted softmax confidence
    confidence = float(
        y_prob[idx, int(y_pred[idx])]
    )

    # Display image
    ax.imshow(image)

    ax.set_title(
        f"True: {true_label}\n"
        f"Pred: {predicted_label} ({confidence:.1%})",
        fontsize=10
    )

    ax.axis("off")

# Hide unused subplots
for ax in axes[len(priority_indices):]:
    ax.axis("off")

# ------------------------------------------------------------
# 4. Save visualization
# ------------------------------------------------------------

fig.suptitle(
    "EcoVision — Targeted Validation Errors",
    fontsize=16,
    fontweight="bold"
)

fig.tight_layout(rect=(0, 0, 1, 0.97))

error_plot_path = (
    experiment_dir / "targeted_error_analysis.png"
)

fig.savefig(
    error_plot_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------

assert error_plot_path.is_file(), (
    "Error analysis image was not saved"
)

print("\nSaved:", error_plot_path)
print("Targeted error analysis: COMPLETED")


## 14. Controlled MobileNetV2 Fine-Tuning

Initialize fine-tuning from the verified best frozen-baseline
checkpoint. Unfreeze only the final 20 backbone layers while
keeping BatchNormalization layers frozen.

Use a lower learning rate and batch size of 4.

In [ ]:

# ============================================================
# 14. Controlled Fine-Tuning Preparation
# ============================================================

from datetime import datetime
import tensorflow as tf

FINE_TUNE_LR = 1e-5
FINE_TUNE_BATCH_SIZE = 4
FINE_TUNE_EPOCHS = 8
UNFREEZE_LAST_N = 20

# Load the verified frozen baseline checkpoint
fine_model = tf.keras.models.load_model(
    str(best_model_path)
)

# Find the MobileNetV2 backbone
fine_base = next(
    layer for layer in fine_model.layers
    if isinstance(layer, tf.keras.Model)
    and layer.name.startswith("mobilenetv2")
)

# Enable selective fine-tuning
fine_base.trainable = True

for layer in fine_base.layers:
    layer.trainable = False

for layer in fine_base.layers[-UNFREEZE_LAST_N:]:
    if not isinstance(
        layer, tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True

# Compile again after changing trainable layers
fine_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FINE_TUNE_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

# Rebuild datasets with a smaller batch size
original_batch_size = BATCH_SIZE

try:
    BATCH_SIZE = FINE_TUNE_BATCH_SIZE

    fine_train_ds = create_dataset(
        train_rows,
        training=True
    )

    fine_val_ds = create_dataset(
        val_rows,
        training=False
    )
finally:
    BATCH_SIZE = original_batch_size

# Apply augmentation to training only
fine_train_ds = fine_train_ds.map(
    augment_batch,
    num_parallel_calls=1
).prefetch(1)

# Separate experiment directory
fine_run_id = datetime.now().strftime(
    "mobilenetv2_finetune_%Y%m%d_%H%M%S"
)

fine_dir = (
    repo_root / "ml" / "reports" / "generated"
    / "week2" / fine_run_id
)

fine_dir.mkdir(parents=True, exist_ok=True)

fine_checkpoint = fine_dir / "best_model.keras"
fine_history_path = fine_dir / "training_history.csv"

fine_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(fine_checkpoint),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(fine_history_path)
    )
]

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in fine_model.trainable_weights
)

print("Fine-tuning experiment:", fine_run_id)
print("Learning rate:", FINE_TUNE_LR)
print("Batch size:", FINE_TUNE_BATCH_SIZE)
print("Trainable parameters:", trainable_params)
print("Training batches:", fine_train_ds.cardinality().numpy())
print("Validation batches:", fine_val_ds.cardinality().numpy())
print("Checkpoint:", fine_checkpoint)
print("Fine-tuning preparation: READY")


## 15. Controlled Fine-Tuning Training

Fine-tune selected MobileNetV2 layers using the verified
frozen-baseline checkpoint.

- Learning rate: 1e-5
- Batch size: 4
- Maximum epochs: 8
- EarlyStopping and ModelCheckpoint enabled
- Validation loss used for checkpoint selection
- Test split remains untouched

In [ ]:

# ============================================================
# 15. Controlled MobileNetV2 Fine-Tuning
# ============================================================

# Confirm the model and datasets are ready
assert fine_base.trainable is True
assert trainable_params > 7686
assert fine_train_ds.cardinality().numpy() == 442
assert fine_val_ds.cardinality().numpy() == 95

print("Starting fine-tuning...")
print("Experiment:", fine_run_id)
print("Learning rate:", FINE_TUNE_LR)
print("Batch size:", FINE_TUNE_BATCH_SIZE)

# Train selected MobileNetV2 layers
fine_history = fine_model.fit(
    fine_train_ds,
    validation_data=fine_val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks,
    verbose=1
)

# ============================================================
# Training summary
# ============================================================

completed_epochs = len(fine_history.history["loss"])

best_loss = min(
    fine_history.history["val_loss"]
)

best_accuracy = max(
    fine_history.history["val_accuracy"]
)

best_epoch = (
    fine_history.history["val_loss"].index(best_loss) + 1
)

print("\n=== Fine-Tuning Summary ===")
print("Epochs executed:", completed_epochs)
print("Best checkpoint epoch:", best_epoch)
print(f"Best validation loss: {best_loss:.6f}")
print(f"Maximum validation accuracy: {best_accuracy:.4%}")

assert fine_checkpoint.is_file(), "Checkpoint not saved"
assert fine_history_path.is_file(), "History not saved"

print("\nCheckpoint:", fine_checkpoint)
print("History:", fine_history_path)
print("Fine-tuning training: COMPLETED")


## 16. Fine-Tuning Checkpoint Evaluation

Reload the best fine-tuning checkpoint and compare its validation
performance against the frozen MobileNetV2 baseline.

Metrics: Accuracy, Macro Precision, Macro Recall, Macro-F1,
per-class performance, and Trash Recall.

Model selection is based on validation loss.
The test split remains sealed.

In [ ]:
    
# ============================================================
# 16. Fine-Tuning Checkpoint Evaluation
# ============================================================

import csv
import json
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Verify saved experiment files
# ------------------------------------------------------------

assert fine_checkpoint.is_file(), "Fine-tuning checkpoint missing"
assert fine_history_path.is_file(), "Fine-tuning history missing"

with open(fine_history_path, "r", newline="") as f:
    fine_records = list(csv.DictReader(f))

with open(history_path, "r", newline="") as f:
    baseline_records = list(csv.DictReader(f))

assert fine_records, "Fine-tuning history is empty"

fine_best_idx = min(
    range(len(fine_records)),
    key=lambda i: float(fine_records[i]["val_loss"])
)

print("=== Fine-Tuning History ===")
print("Epochs recorded:", len(fine_records))
print("Best checkpoint epoch:", fine_best_idx + 1)
print(
    "Best recorded validation loss:",
    fine_records[fine_best_idx]["val_loss"]
)

# ------------------------------------------------------------
# 2. Reload best fine-tuning checkpoint
# ------------------------------------------------------------

fine_best = tf.keras.models.load_model(
    str(fine_checkpoint)
)

fine_val_metrics = fine_best.evaluate(
    fine_val_ds,
    return_dict=True,
    verbose=1
)

# ------------------------------------------------------------
# 3. Predict on validation only
# ------------------------------------------------------------

fine_probs = fine_best.predict(
    fine_val_ds,
    verbose=1
)

fine_pred = np.argmax(
    fine_probs,
    axis=1
)

assert fine_probs.shape == (len(val_rows), NUM_CLASSES)
assert len(y_true) == len(fine_pred)
assert np.all(np.isfinite(fine_probs))

np.testing.assert_allclose(
    fine_probs.sum(axis=1),
    np.ones(len(fine_probs)),
    atol=1e-5
)

# ------------------------------------------------------------
# 4. Calculate comparable metrics
# ------------------------------------------------------------

def calculate_metrics(true_labels, predictions):
    class_recall = recall_score(
        true_labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        average=None,
        zero_division=0
    )

    return {
        "accuracy": float(
            accuracy_score(true_labels, predictions)
        ),
        "macro_precision": float(
            precision_score(
                true_labels, predictions,
                average="macro", zero_division=0
            )
        ),
        "macro_recall": float(
            recall_score(
                true_labels, predictions,
                average="macro", zero_division=0
            )
        ),
        "macro_f1": float(
            f1_score(
                true_labels, predictions,
                average="macro", zero_division=0
            )
        ),
        "trash_recall": float(
            class_recall[label_to_id["trash"]]
        )
    }

baseline_metrics = calculate_metrics(
    y_true, y_pred
)

fine_metrics = calculate_metrics(
    y_true, fine_pred
)

baseline_metrics["val_loss"] = min(
    float(r["val_loss"]) for r in baseline_records
)

fine_metrics["val_loss"] = float(
    fine_val_metrics["loss"]
)

# Check alignment with Keras accuracy
np.testing.assert_allclose(
    fine_metrics["accuracy"],
    fine_val_metrics["accuracy"],
    atol=1e-5
)

# ------------------------------------------------------------
# 5. Print experiment comparison
# ------------------------------------------------------------

print("\n=== Baseline vs Fine-Tuning ===")
print(f"{'Metric':<20} {'Baseline':>12} {'Fine-Tuning':>14}")
print("-" * 48)

for metric in [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "trash_recall",
    "val_loss"
]:
    print(
        f"{metric:<20} "
        f"{baseline_metrics[metric]:>12.4f} "
        f"{fine_metrics[metric]:>14.4f}"
    )

# ------------------------------------------------------------
# 6. Per-class classification report
# ------------------------------------------------------------

print("\n=== Fine-Tuning Per-Class Report ===")

print(
    classification_report(
        y_true,
        fine_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

fine_cm = confusion_matrix(
    y_true,
    fine_pred,
    labels=list(range(NUM_CLASSES))
)

print("\n=== Fine-Tuning Confusion Matrix ===")
print(fine_cm)

print("\nCorrect predictions:", np.trace(fine_cm))
print("Incorrect predictions:", fine_cm.sum() - np.trace(fine_cm))

# ------------------------------------------------------------
# 7. Save evaluation summary
# ------------------------------------------------------------

comparison = {
    "baseline": baseline_metrics,
    "fine_tuning": fine_metrics,
    "fine_tuning_best_epoch": fine_best_idx + 1,
    "validation_samples": len(val_rows),
    "selection_metric": "val_loss",
}

comparison_path = (
    fine_dir / "validation_comparison.json"
)

with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

print("\nSaved:", comparison_path)
print("Fine-tuning checkpoint evaluation: COMPLETED")


## 17. Baseline vs Fine-Tuning Error Comparison

Compare prediction changes between the two checkpoints
on the same validation images.

Focus on the minority trash class and changes in
per-class recall. The test split remains sealed.

In [ ]:

# ============================================================
# 17. Baseline vs Fine-Tuning Error Comparison
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Compare predictions of the trash class
trash_id = label_to_id["trash"]

trash_indices = np.where(y_true == trash_id)[0]

baseline_correct = (
    y_pred[trash_indices] == trash_id
)

fine_correct = (
    fine_pred[trash_indices] == trash_id
)

both_correct = np.sum(
    baseline_correct & fine_correct
)

baseline_only = np.sum(
    baseline_correct & ~fine_correct
)

fine_only = np.sum(
    ~baseline_correct & fine_correct
)

both_wrong = np.sum(
    ~baseline_correct & ~fine_correct
)

print("=== Trash Prediction Comparison ===")
print("Total trash images:", len(trash_indices))
print("Both models correct:", both_correct)
print("Baseline only correct:", baseline_only)
print("Fine-tuning only correct:", fine_only)
print("Both models incorrect:", both_wrong)

print("\n=== Trash Prediction Distribution ===")

for name, matrix in [
    ("Baseline", cm),
    ("Fine-tuning", fine_cm)
]:
    print(f"\n{name}:")
    for label, count in zip(
        CLASS_NAMES,
        matrix[trash_id]
    ):
        print(f"  Predicted {label}: {count}")

# ------------------------------------------------------------
# Visualize both confusion matrices
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

for ax, matrix, title in zip(
    axes,
    [cm, fine_cm],
    ["Frozen Baseline", "Fine-Tuning"]
):
    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=CLASS_NAMES
    )

    display.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False,
        xticks_rotation=45
    )

    ax.set_title(title)

fig.suptitle(
    "EcoVision — Baseline vs Fine-Tuning",
    fontsize=15
)

plt.tight_layout()

comparison_plot_path = (
    fine_dir / "baseline_vs_finetuning_cm.png"
)

fig.savefig(
    comparison_plot_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nSaved:", comparison_plot_path)
print("Comparison analysis: COMPLETED")


## 18. Week 2 Model Selection

Document the comparison between the frozen baseline
and the controlled fine-tuning experiment.

The frozen baseline is provisionally selected based on
macro-F1 and minority-class recall, while the fine-tuned
checkpoint is retained for reproducibility.

In [ ]:

# ============================================================
# 18. Week 2 Model Selection Report
# ============================================================

from pathlib import Path

report_path = repo_root / "ml/reports/week2_model_selection.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

def pct(value):
    return f"{value * 100:.2f}%"

report = f"""# EcoVision — Week 2 Model Selection

## Experiment overview

Dataset: TrashNet (six classes)
Training samples: {len(train_rows)}
Validation samples: {len(val_rows)}
Test split: Reserved, not evaluated

Backbone: MobileNetV2 pretrained on ImageNet

## Validation results

| Metric | Frozen baseline | Fine-tuning |
|---|---:|---:|
| Accuracy | {pct(baseline_metrics['accuracy'])} | {pct(fine_metrics['accuracy'])} |
| Macro Precision | {pct(baseline_metrics['macro_precision'])} | {pct(fine_metrics['macro_precision'])} |
| Macro Recall | {pct(baseline_metrics['macro_recall'])} | {pct(fine_metrics['macro_recall'])} |
| Macro F1 | {pct(baseline_metrics['macro_f1'])} | {pct(fine_metrics['macro_f1'])} |
| Trash Recall | {pct(baseline_metrics['trash_recall'])} | {pct(fine_metrics['trash_recall'])} |
| Validation Loss | {baseline_metrics['val_loss']:.6f} | {fine_metrics['val_loss']:.6f} |

## Checkpoints

Frozen baseline: `{best_model_path.relative_to(repo_root)}`

Fine-tuning: `{fine_checkpoint.relative_to(repo_root)}`

Both checkpoints are local generated artifacts and
are not included in the Git repository.

## Model selection

Selected candidate: Frozen MobileNetV2 baseline, epoch 8.

Rationale:
- Higher macro-F1 than the fine-tuned model.
- Higher recall for the minority trash class.
- Fine-tuning improved overall accuracy and loss,
  but reduced trash recall.
- The fine-tuning experiment remains a comparator.

## Known limitations

- TrashNet has class imbalance.
- Trash is a heterogeneous category.
- Validation results may not represent smartphone photos.
- No independent test-set evaluation has been performed.
- No TensorFlow Lite parity or Android latency results exist yet.

## Next steps

- Preserve both checkpoints and experiment histories.
- Complete reproducibility documentation.
- Continue with model export and final evaluation planning.
"""

report_path.write_text(report, encoding="utf-8")

assert report_path.is_file()
print("Saved:", report_path)
print("Week 2 model selection report: COMPLETED")
